# Single-Bus Substation

The single-bus configuration is the simplest possible design, with the lowest cost and also the lowest reliability. All equipment and switching devices are connected to a single main bus, which is always energized, as shown below. This configuration is commonly used in small electric distribution substations and wind farm collectors due to its simplicity and lower cost. Bus faults and breaker failures will result in an outage or significant voltage deviation to all branches in the entire substation as all branches share a common bus that will propagate the problem to said branches

## CIM Representation

In CIM, all buses and junctions are represented by the ConnectivityNode class. If a node corresponds to a bus bar, then a BusBarSection object is appended to the ConnectivityNode, as shown below in Figure 2. The single-bus configuration only includes a single main bus, with all distribution feeders connected to said bus. Full node-breaker switch representation adds a set of one Breaker and two Disconnector objects for each feeder added to the substation.

![single-bus](../images/single_bus.png)

----

## SingleBusSubtation

* new_branch

In [1]:
# Import cimgraph modules
from cimgraph.models import FeederModel, NodeBreakerModel
from cimgraph.databases import RDFlibConnection, XMLFile
import cimgraph.utils as utils
import cimgraph.data_profile.cimhub_2023 as cim

In [2]:
# Set environment variables
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'

In [3]:
connection = XMLFile(filename='new_single_bus.xml')

In [4]:
from cimbuilder.substation_builder import SingleBusSubstation

In [5]:
sub_builder = SingleBusSubstation(connection=connection, name="single_bus_sub", base_voltage=115000)
substation = sub_builder.substation


In [8]:
# Import 13 bus model from XML file
ieee13_feeder = cim.Feeder(mRID = '49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
xml13 = XMLFile(filename='../../tests/test_models/IEEE13.xml')
ieee13_network = FeederModel(connection=xml13, container=ieee13_feeder, distributed=False)

In [9]:
sub_builder.new_feeder(breaker_number= 10, feeder=ieee13_feeder, feeder_network=ieee13_network)


In [12]:

assets13_feeder = cim.Feeder(mRID = '5B816B93-7A5F-B64C-8460-47C17D6E4B0F')
xml13assets = XMLFile(filename='../../tests/test_models/IEEE13_Assets.xml')
assets13_network = FeederModel(connection=xml13assets, container=assets13_feeder, distributed=False)

In [13]:
sub_builder.new_feeder(breaker_number = 20, feeder=assets13_feeder, feeder_network=assets13_network)

In [14]:
sub_builder.network.pprint(cim.Substation)

[
    {
        "@id": "09120c85-1d5b-422d-bc1e-6cc0039e04c8",
        "@type": "Substation",
        "name": "single_bus_sub",
        "NormalEnergizedFeeder": [
            {
                "@id": "49ad8e07-3bf9-a4e2-cb8f-c3722f837b62",
                "@type": "Feeder"
            },
            {
                "@id": "5b816b93-7a5f-b64c-8460-47c17d6e4b0f",
                "@type": "Feeder"
            }
        ]
    }
]


In [15]:
sub_builder.upload()

In [16]:
sub_builder.pprint(cim.Feeder)

[
    {
        "@id": "49ad8e07-3bf9-a4e2-cb8f-c3722f837b62",
        "@type": "Feeder",
        "NormalEnergizingSubstation": {
            "@id": "09120c85-1d5b-422d-bc1e-6cc0039e04c8",
            "@type": "Substation"
        }
    },
    {
        "@id": "5b816b93-7a5f-b64c-8460-47c17d6e4b0f",
        "@type": "Feeder",
        "NormalEnergizingSubstation": {
            "@id": "09120c85-1d5b-422d-bc1e-6cc0039e04c8",
            "@type": "Substation"
        }
    }
]


In [17]:
sub_builder.write_json_ld(filename='new_single_bus.json')

## Round-Trip Test: Load and Read from Database

### Delete all old entries from database and load new models

In [ ]:
from cimloader.uploaders import BlazegraphUploader
loader = BlazegraphUploader()
loader.drop_all()

In [ ]:
loader.upload_from_xml(filename='../../sample_models/ieee13.xml')
loader.upload_from_xml(filename='../../sample_models/ieee13_assets.xml')
loader.upload_from_xml(filename='new_single_bus.xml')

In [ ]:
# Connect to Blazegraph Database
from cimgraph.databases import BlazegraphConnection
blazegraph = BlazegraphConnection()
network = NodeBreakerModel(container=substation, connection=blazegraph, distributed = False)

In [ ]:
# Print substation info
network.get_all_edges(cim.Substation)
network.pprint(cim.Substation)

In [ ]:
# Print feeder info
network.get_all_edges(cim.Feeder)
network.pprint(cim.Feeder)

In [ ]:
# Print total load served by substation from both feeders
total_load = 0
network.get_all_edges(cim.EnergyConsumer)
for load in network.graph[cim.EnergyConsumer].values():
    total_load = total_load + float(load.p)

print(f'total load is {total_load/1000} kW')

In [ ]:
utils.get_all_data(network)

In [ ]:
utils.write_xml(network, 'single_bus_and_feeders.xml')